# 🧪 W10-D4 Fail-closed、Fail-open 与 Approval

> 配套阅读：同名 `.md`。本 notebook 只用小规模、可重复的模拟来验证核心治理约束。

**实验目标：** 把安全边界和执行故障放入同一策略矩阵，并模拟多审批人阈值。


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

scenarios = [
    ("权限 scope 不匹配", "security", False),
    ("Prompt hash 不匹配", "security", False),
    ("LLM 超时", "execution", True),
    ("知识库临时不可用", "execution", True),
]

def decide(layer, can_fallback):
    if layer == "security": return "FAIL_CLOSED: 拒绝执行"
    return "FALLBACK: 返回风险标记" if can_fallback else "FAIL_CLOSED"

for name, layer, fallback in scenarios:
    print(f"{name:16s} -> {decide(layer, fallback)}")


In [ ]:
# 兜底不是“假装成功”：敏感输入仍进入人工复核。
def fallback_result(message):
    sensitive = any(word in message.lower() for word in ["退款", "解约", "减免"])
    return {"summary": "系统暂时无法完成", "human_review_required": sensitive,
            "risk_flags": ["sensitive_topic"] if sensitive else []}

for message in ["查询本月客流", "我要申请退款"]:
    print(message, "->", fallback_result(message))

# 超时默认不批准；审批必须有明确达标票数。
def approval(votes, mode):
    yes, no = votes.count("yes"), votes.count("no")
    if mode == "single": return "approved" if yes else ("rejected" if no else "pending")
    if mode == "unanimous": return "rejected" if no else ("approved" if yes == len(votes) else "pending")
    return "approved" if yes > len(votes)/2 else ("rejected" if no >= len(votes)/2 else "pending")

for mode in ["single", "majority", "unanimous"]:
    print(mode, "[yes, no, yes] ->", approval(["yes", "no", "yes"], mode))


In [ ]:
modes = ["single", "majority", "unanimous"]
ballots = [["yes"], ["yes", "no", "yes"], ["yes", "yes", "yes"]]
results = [approval(v, m) for m, v in zip(modes, ballots)]
colors = ["#59A14F" if r == "approved" else "#E15759" for r in results]
plt.figure(figsize=(6, 3.2))
plt.bar(modes, [1 if r == "approved" else 0 for r in results], color=colors)
plt.yticks([0, 1], ["未批准", "批准"]); plt.title("不同审批策略的阈值结果")
plt.tight_layout(); plt.show()
print("关键默认值：审批超时 -> pending，不自动批准。")
